In [1]:
from langgraph.graph import StateGraph, START,END
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import tool
import requests 


from langgraph.graph.message import add_messages


In [2]:
load_dotenv()

True

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
) 

model = ChatHuggingFace(llm = llm)

In [4]:
search_tools = DuckDuckGoSearchRun(region = "us-en")

@tool
def calculator(first_num:float, second_num: float, operation :str) -> dict:
    """
    Perform a basic airthrmetic operation on two number.
    Supported operations : add, sub, mul , div
    """

    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num*second_num
        elif operation == "div":
            if second_num == 0:
                return {"error" : "ZeroDivisionError"}      
            result = first_num/ second_num
        else:          
            return {"error" : f"unsupported operation :{operation}"}
        
        return {"first_num" : first_num, "second_num" : second_num, "operation" : operation, "result" : result}
    except Exception as e:
        return {"error" : str(e)}
    
@tool
def get_stock_price(symbol:str)-> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPl', 'TSLS')
    using Alpha Vanatge with API key in the URL.
    """

    url = "https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=BINZ2Q5WA6NMREMV"
    r = requests.get(url)
    return r.json


In [5]:
tools = [search_tools,get_stock_price,calculator]
llm_tools = model.bind_tools(tools)

In [6]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [7]:
def chat_node (state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_tools.invoke(messages)
    return {"messages" : [response]}

tool_node = ToolNode(tools)


In [8]:
graph = StateGraph(ChatState)

graph.add_node("chat_node" , chat_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node" , tools_condition)
graph.add_edge("tools" , "chat_node")



In [9]:
app = graph.compile()

In [ ]:
from langchain_core.messages import SystemMessage

result = app.invoke({
    "messages": [
        SystemMessage(content="You are a helpful AI that MUST use tools for calculations."),
        HumanMessage(content="What is 2*3?")
    ]
})

print(result["messages"][-1].content)